## Reasoning Patch

In [1]:
%load_ext autoreload
%autoreload 2

### Overview

Patches attention-head outputs between a base and source prompt — sweeping over whole layers and then over individual heads — to measure each head's/layer's causal effect on the model's final answer. Only the last-token (prediction) position is patched.

### Set-up

Imports `_config`/`_dataset`/`_prompt`/`_mapping` plus the intervention primitives used for the layer- and head-patching sweeps below.

In [2]:
import torch
import gc
import random
import pandas as pd
from tqdm import tqdm

import sys
sys.path.append("src")
import _config
import _dataset
import _prompt
import _mapping
from _intervention import prepare_batch_multitoken_intervention, batch_intervene, get_attention_freeze_hooks, prepare_batch_multitoken_head_intervention

## Experiment Config

In [3]:
prompt_config = _config.PromptConfig(
    model_type="GPT-OSS_stepwise", # GPT-OSS or R1
    prompt_type="h_pre_penultimate_sum", # {null / h / h1 / h2}_{null / pre_result / pre_final_sum / ...}
)
intervention_config = _config.InterventionConfig(
    intervention_loc="", # restatement or reasoning or restatement_and_reasoning
    intervention_ids=[20],
    tok_pos_fn=_mapping.intervene_id_to_tok_pos_stepwise_3_digit_h,
    module_format="model.layers.{layer}.self_attn",
    pre_hook=False,
)
attention_config = _config.AttentionFreezeConfig(
    enabled=False,
    num_attention=20,
    dataset_fn=_dataset.create_h_dataset,
    num_digits=3,
    prompt_fn=_prompt.get_stepwise_prompt,
    divide_num=22,
)
run_config = _config.RunConfig(result_dir="layerwise")

tok_pos_fn = intervention_config.tok_pos_fn
prompt_fn = attention_config.prompt_fn
num_attention = attention_config.num_attention
freeze_attention = attention_config.enabled
modifier_fn = lambda prompt, add_ds_entry: prompt

## Set up Experiment

Loads the model/tokenizer, builds the (here disabled) attention-freeze hooks, and loads the configured base/source prompt pairs to patch.

In [4]:
model, tokenizer = _config.load_model(prompt_config.model_type)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
def intervene_on_final_sum(prompt, add_ds_entry):
    source_prompt = attention_config.prompt_fn(
        add_ds_entry["source_1_digits"],
        add_ds_entry["source_2_digits"],
        add_ds_entry["source_1_num"],
        add_ds_entry["source_2_num"],
    )
    return _prompt.get_intervened_prompt(intervention_ids, prompt, source_prompt)

modifier_fn = intervene_on_final_sum

In [5]:
attention_freeze_hooks = _config.build_attention_freeze_hooks(
    model,
    tokenizer,
    attention_config,
    modifier_fn=modifier_fn,
)

In [6]:
prompts = _config.load_prompts(prompt_config)
print(f"loaded {len(prompts)} prompts")

loaded 256 prompts


In [7]:
intervention_ids = _config.resolve_intervention_ids(
    prompt_config.model_type,
    prompt_config.prompt_type,
    intervention_config.intervention_loc,
    intervention_config.intervention_ids,
)
print(intervention_ids)

[20]


In [9]:
# for i, row in prompts.iterrows():
#     print(list(enumerate(tokenizer.convert_ids_to_tokens(tokenizer(row["base_prompt"], add_special_tokens=False, return_tensors="pt")["input_ids"][0]))))

In [8]:
# Only the last-token position is patched: that is the position whose attention-
# head output feeds directly into the next-token prediction.
tok_pos_list = [intervention_config.tok_pos_fn[attention_config.divide_num] - 1]
print(tok_pos_list)

[235]


## Run Experiment

For each layer, patches that layer's attention output — with and without attention freezing — and then sweeps over every individual head, each time recording the effect on the factual vs counterfactual answer probability at the last-token position.

In [9]:
if freeze_attention:

    header = list(prompts.columns) + ['intervention_ids', 'intervention_id', 'layer', 'generated_text', 'factual_label_probability', 'counterfactual_label_probability']
    freeze_run_config = _config.RunConfig(
        experiment_root=run_config.experiment_root,
        result_dir=run_config.result_dir,
        output_filename=f"{prompt_config.stem}{intervention_config.location_suffix}.csv",
    )
    filepath = _config.build_run_output_filepath(prompt_config, freeze_run_config, header)

    # Batch size 1, because we run 20 attention patterns at once
    for i, row in tqdm(prompts.iterrows(), total=len(prompts)):
        # Preparing prompts and labels
        base_prompt = row['base_prompt']
        base_prompts = [row['base_prompt']] * len(attention_prompts)
        if "factual_output" in row and pd.notna(row["factual_output"]):
            factual_labels_str = [str(row['factual_output'])] * len(attention_prompts)
        else:
            factual_labels_str = [str(row['base_sum'])] * len(attention_prompts)
        factual_labels = tokenizer(factual_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze(1)
        source_prompt = row['source_prompt']
        source_prompts = [row['source_prompt']] * len(attention_prompts)
        if "counterfactual_output" in row and pd.notna(row["counterfactual_output"]):
            counterfactual_labels_str = [str(row['counterfactual_output'])] * len(attention_prompts)
        else:
            counterfactual_labels_str = [str(row['source_sum'])] * len(attention_prompts)
        counterfactual_labels = tokenizer(counterfactual_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze(1)

        # Preparing intervention hooks
        for layer in range(len(model.model.layers)):
            for head in range(64):
                intervene_hooks = []
                tokens, source_tokens, hook = prepare_batch_multitoken_intervention(model, tokenizer, layer, tok_pos_list, base_prompts, source_prompts, pre_hook=False, module_format=f"model.layers.{layer}.self_attn")
                intervene_hooks.append(hook)
                input_length = tokens["input_ids"].shape[1]

                # Forward pass
                with torch.no_grad():
                    output = batch_intervene(model, tokens["input_ids"], attention_freeze_hooks + intervene_hooks, attention_mask=tokens["attention_mask"], pre_hook=False)
                pred_toks = output.logits[:,-1,:].argmax(dim=-1)
                prob = torch.nn.functional.softmax(output.logits[:,-1,:], dim=-1)
                factual_prob = prob[torch.arange(prob.shape[0]), factual_labels]
                counterfactual_prob = prob[torch.arange(prob.shape[0]), counterfactual_labels]
                tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)
                del output
                
                # Writing results
                for j in range(len(attention_prompts)):
                    generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
                    _config.write_to_csv(filepath, row.to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), layer, generated_text.replace(tokenizer.pad_token[-1], ""), factual_prob[j].item(), counterfactual_prob[j].item()])

                del tokens, source_tokens, intervene_hooks
                torch.cuda.empty_cache()
                gc.collect()

else:
    header = list(prompts.columns) + ['intervention_ids', 'intervention_id', 'layer', 'generated_text', 'factual_label_probability', 'counterfactual_label_probability']
    no_freeze_run_config = _config.RunConfig(
        experiment_root=run_config.experiment_root,
        result_dir=run_config.result_dir,
        output_filename=f"{prompt_config.stem}_{intervention_config.location_suffix}.csv",
    )
    filepath = _config.build_run_output_filepath(prompt_config, no_freeze_run_config, header)

    batch_size = 24
    for i in tqdm(range(0, len(prompts), batch_size)):
        # Preparing prompts and labels
        batch_rows = prompts.iloc[i:i+batch_size]
        if pd.notna(batch_rows.iloc[0]['factual_output']) and batch_rows.iloc[0]['factual_output']:
            factual_labels_str = [str(sum) for sum in batch_rows['factual_output'].tolist()]
        else:
            factual_labels_str = [str(sum) for sum in batch_rows['base_sum'].tolist()]
        factual_labels = tokenizer(factual_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"]
        factual_labels = factual_labels.squeeze(1)
        if pd.notna(batch_rows.iloc[0]['counterfactual_output']) and batch_rows.iloc[0]['counterfactual_output']:
            counterfactual_labels_str = [str(sum) for sum in batch_rows['counterfactual_output'].tolist()]
        else:
            counterfactual_labels_str = [str(sum) for sum in batch_rows['source_sum'].tolist()]
        counterfactual_labels = tokenizer(counterfactual_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"]
        counterfactual_labels = counterfactual_labels.squeeze(1)

        # Preparing intervention hooks
        for layer in range(len(model.model.layers)):
            intervene_hooks = []
            tokens, source_tokens, hook = prepare_batch_multitoken_intervention(model, tokenizer, layer, tok_pos_list, batch_rows['base_prompt'].tolist(), batch_rows['source_prompt'].tolist(), module_format=f"model.layers.{layer}.self_attn", pre_hook=False)
            intervene_hooks.append(hook)
            input_length = tokens["input_ids"].shape[1]

            # Forward pass
            with torch.no_grad():
                output = batch_intervene(model, tokens["input_ids"], intervene_hooks, attention_mask=tokens["attention_mask"], pre_hook=False)
            pred_toks = output.logits[:,-1,:].argmax(dim=-1)
            prob = torch.nn.functional.softmax(output.logits[:,-1,:], dim=-1)
            factual_prob = prob[torch.arange(prob.shape[0]), factual_labels]
            counterfactual_prob = prob[torch.arange(prob.shape[0]), counterfactual_labels]
            tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)
            del output
            
            # Writing results
            for j, (_, row) in enumerate(batch_rows.iterrows()):
                generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
                _config.write_to_csv(filepath, row.to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), layer, generated_text.replace(tokenizer.pad_token[-1], ""), factual_prob[j].item(), counterfactual_prob[j].item()])

            del tokens, source_tokens, intervene_hooks
            torch.cuda.empty_cache()
            gc.collect()

  0%|                                                                                                                        | 0/11 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [46:27<00:00, 253.42s/it]


In [ ]:
if freeze_attention:
    batch_size = 1
else:
    batch_size = 24

header = list(prompts.columns) + ['intervention_ids', 'intervention_id', 'layer', 'generated_text', 'factual_label_probability', 'counterfactual_label_probability']
per_head_run_config = _config.RunConfig(
    experiment_root=run_config.experiment_root,
    result_dir=run_config.result_dir,
    output_filename=f"{intervention_config.location_suffix}_{prompt_config.stem}.csv",
)
filepath = _config.build_run_output_filepath(prompt_config, per_head_run_config, header)

for i in tqdm(range(0, len(prompts), batch_size)):
    # Preparing prompts and labels
    batch_rows = prompts.iloc[i:i+batch_size]
    if pd.notna(batch_rows.iloc[0]['factual_output']) and batch_rows.iloc[0]['factual_output']:
        factual_labels_str = [str(sum) for sum in batch_rows['factual_output'].tolist()]
    else:
        factual_labels_str = [str(sum) for sum in batch_rows['base_sum'].tolist()]
    factual_labels = tokenizer(factual_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"]
    factual_labels = factual_labels.squeeze(1)
    if pd.notna(batch_rows.iloc[0]['counterfactual_output']) and batch_rows.iloc[0]['counterfactual_output']:
        counterfactual_labels_str = [str(sum) for sum in batch_rows['counterfactual_output'].tolist()]
    else:
        counterfactual_labels_str = [str(sum) for sum in batch_rows['source_sum'].tolist()]
    counterfactual_labels = tokenizer(counterfactual_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"]
    counterfactual_labels = counterfactual_labels.squeeze(1)

    base_prompts = batch_rows['base_prompt'].tolist()
    source_prompts = batch_rows['source_prompt'].tolist()

    # If freeze_attention, repeat prompts and labels 20 times
    if freeze_attention:
        base_prompts = base_prompts * 20
        source_prompts = source_prompts * 20
        factual_labels = factual_labels.repeat(20)
        counterfactual_labels = counterfactual_labels.repeat(20)

    for layer in range(len(model.model.layers)):
        for head in range(64):
            # Preparing intervention hooks
            intervene_hooks = []
            tokens, source_tokens, hooks = prepare_batch_multitoken_head_intervention(model, tokenizer, [layer], tok_pos_list, base_prompts, source_prompts, head_idx=head)
            intervene_hooks += hooks
            input_length = tokens["input_ids"].shape[1]

            # Forward pass
            with torch.no_grad():
                output = batch_intervene(model, tokens["input_ids"], intervene_hooks+attention_freeze_hooks, attention_mask=tokens["attention_mask"])
            pred_toks = output.logits[:,-1,:].argmax(dim=-1)
            prob = torch.nn.functional.softmax(output.logits[:,-1,:], dim=-1)
            factual_prob = prob[torch.arange(prob.shape[0]), factual_labels]
            counterfactual_prob = prob[torch.arange(prob.shape[0]), counterfactual_labels]
            tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)
            del output
            
            # Writing results
            if freeze_attention:
                for j in range(num_attention):
                    generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
                    _config.write_to_csv(filepath, batch_rows.iloc[0].to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), generated_text.replace(tokenizer.pad_token[-1], ""), factual_prob[j].item(), counterfactual_prob[j].item()])
            else:
                for j in range(batch_size):
                    generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
                    _config.write_to_csv(filepath, batch_rows.iloc[j].to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), generated_text.replace(tokenizer.pad_token[-1], ""), factual_prob[j].item(), counterfactual_prob[j].item()])

            del tokens, source_tokens, intervene_hooks
            torch.cuda.empty_cache()
            gc.collect()